# 課程 05 - 主體性檢索增強生成模型 (Agentic RAG)


## 設定

此筆記本展示使用 Microsoft Agent Framework 的 Agentic RAG（檢索增強生成）模式。

**先決條件：**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — 您的 Azure AI Search 服務端點
- `AZURE_SEARCH_API_KEY` — 您的 Azure AI Search API 密鑰
- 透過環境變數配置的 Azure OpenAI 部署
- 已透過 Azure CLI 認證（`az login`）


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什麼是 Agentic RAG？

傳統的 RAG 遵循固定流程：先檢索文件，然後生成回答。**Agentic RAG** 更進一步，賦予代理自主權決定**何時**以及**如何**檢索資訊。

使用 Agentic RAG，代理可以：
- **決定**在回答問題之前是否需要檢索
- **選擇**要查詢的數據來源或工具
- **評估**檢索結果，若第一次嘗試不足，則執行後續檢索
- **結合**多次檢索步驟中的資訊，形成連貫的答案

這使代理相比靜態的檢索-生成流程更具彈性和準確性。


## 建立搜尋工具

在 Agentic RAG 中，外部資料來源被包裝為代理人可按需調用的**工具**。這讓代理人將檢索視為它可以執行的另一種操作，而不是必須的步驟。

以下我們定義一個旅遊知識庫，並將其作為代理人可以呼叫以查詢目的地資訊的工具。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## 建立 RAG 代理人

現在我們建立一個被指示為 **在回答前始終檢索資訊** 的代理人。該代理人使用 `search_travel_knowledge` 工具，將其回應以知識庫為基礎，而非依賴自身的訓練資料。


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 迭代檢索 — 製作者-檢查者模式

Agentic RAG 的一大優勢是**迭代檢索**。代理人可以進行多輪搜尋，以驗證、修正或擴充其初步發現——類似「製作者-檢查者」的工作流程：

1. **製作者步驟**：代理人檢索初步資訊並擬出回應。
2. **檢查者步驟**：代理人進一步檢索，以核實細節或填補空白。

以下，代理人被問及一個需要比較多個目的地的問題，促使其進行多次搜尋。


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 總結

在本課程中，您學會如何使用 Microsoft Agent Framework 建立一個**具代理性的 RAG**系統：

- **具代理性的 RAG**讓代理人能自主決定何時檢索資訊，使檢索動態化，而非固定。
- **以工具作為資料來源**：外部知識庫（如 Azure AI Search）被包裝成代理人可調用的工具。
- **反覆檢索**：製造者-檢查者模式讓代理人得以進行多輪檢索 — 搜尋、驗證並精煉 — 再產出最終答案。

在生產環境中，您會將記憶體中的 `TRAVEL_KNOWLEDGE_BASE` 換成真正的 Azure AI Search 索引，以處理大型旅遊文件檢索。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於翻譯的準確性，但請注意，自動翻譯可能會包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於重要資訊，建議使用專業人工翻譯。我們對因使用本翻譯而產生的任何誤解或曲解概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
